In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    trim,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_tun_profits(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "TunId String, SourceSystem String, PortfolioKey String, PortfolioName String,"
        #     "IngestionTimestamp timestamp, BatchId String, TunKey String, TunCode String, ExposureKey String,"
        #     "DepositKey String, PaymentKey String, LoanAmount decimal(18,2), DepositAmount decimal(18,2),"
        #     "LoanTransactionUnitCode int, LoanTransactionUnit int, DepositTransactionUnitCode int, DepositTransactionUnit int,"
        #     "LoanTransactionUser String, DepositTransactionUser String, LoanTransactionSN int, DepositTransactionSN int,"
        #     "LoanTransactionDate timestamp, DepositTransactionDate timestamp, DepositTransactionSerialNumber int,"
        #     "DepositEntrySerialNumber int, TransactionDateTime timestamp, TimeStampDate timestamp, BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Tun"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "profits",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = [
            "PortfolioName",
            "Prefix",
            "unit_code_profits",
            "businessdate",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================== Tun ==================

        Tun_extra_filters = None

        if Tun_extra_filters is not None:
            Tun_extra_filters.append(date_filter)
            Tun_final_filters = " AND ".join(Tun_extra_filters)
        else:
            Tun_final_filters = date_filter

        Tun_selected_fields = [
            "Account_Number",
            "LNS_TRX_DATE",
            "lns_trx_unit",
            "lns_trx_usr",
            "lns_trx_sn",
            "Tun",
            "DEP_TRX_SN",
            "DEP_TRANS_SER_NUM",
            "DEPOSIT_ACCOUNT_NUMBER",
            "LNS_AMOUNT",
            "DEP_AMOUNT",
            "DEP_TRX_UNIT",
            "DEP_TRX_USR",
            "DEP_TRX_DATE",
            "DEP_ENTRY_SER_NUM",
            "LNS_TRX_TMSTAMP",
            "TMSTAMP",
            "businessdate",
        ]

        Tun = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_Tun",  # check name
            Tun_final_filters,
            Tun_selected_fields,
        )

        # HashKeys
        target_table_profits_TunId_primary_key_fields = [
            Tun.Account_Number,
            Tun.LNS_TRX_DATE,
            Tun.lns_trx_unit,
            Tun.lns_trx_usr,
            Tun.lns_trx_sn,
            Tun.Tun,
            Tun.DEP_TRX_SN,
            Tun.DEP_TRANS_SER_NUM,
            lit("profits"),
        ]

        Tun = (
            Tun.withColumn(
                "TunId",
                sha2(
                    concat_ws(";", *target_table_profits_TunId_primary_key_fields),
                    256,
                ),
            )
            .withColumn(
                "TunCode",
                when(
                    substring(trim(Tun.Tun), 1, 4) != "Tun:",
                    concat(lit("Tun:"), Tun.Tun),
                ).otherwise(Tun.Tun),
            )
            .withColumn(
                "PaymentKey",
                concat_ws(
                    ";",
                    Tun.LNS_TRX_DATE,
                    trim(Tun.lns_trx_unit),
                    trim(Tun.lns_trx_usr),
                    trim(Tun.lns_trx_sn),
                ),
            )
            .withColumn(
                "TunKey",
                concat_ws(
                    ";",
                    Tun.Account_Number,
                    Tun.LNS_TRX_DATE,
                    trim(Tun.lns_trx_unit),
                    trim(Tun.lns_trx_usr),
                    trim(Tun.lns_trx_sn),
                    Tun.Tun,
                    Tun.DEP_TRX_SN,
                    Tun.DEP_TRANS_SER_NUM,
                ),
            )
        )

        Tun_portfolio_join_condition = (
            substring(Tun["Account_Number"], 1, 3) == portfolio["Prefix"]
        )

        target_table_profits = Tun.join(
            portfolio,
            Tun_portfolio_join_condition,
            "left",
        )

        # ================== Selection of Final Columns ==================

        result_df_profits = target_table_profits.select(
            Tun.TunId.alias("TunId"),
            lit("profits").alias("SourceSystem"),
            portfolio.unit_code_profits.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            Tun.TunKey.alias("TunKey"),
            Tun.Tun.alias("TunCode"),
            Tun.Account_Number.alias("ExposureKey"),
            Tun.DEPOSIT_ACCOUNT_NUMBER.alias("DepositKey"),
            Tun.PaymentKey.alias("PaymentKey"),
            Tun.LNS_AMOUNT.cast("Decimal(18,2)").alias("LoanAmount"),
            Tun.DEP_AMOUNT.cast("Decimal(18,2)").alias("DepositAmount"),
            Tun.lns_trx_unit.alias("LoanTransactionUnitCode"),
            Tun.lns_trx_unit.alias("LoanTransactionUnit"),
            Tun.DEP_TRX_UNIT.alias("DepositTransactionUnitCode"),
            Tun.DEP_TRX_UNIT.alias("DepositTransactionUnit"),
            Tun.lns_trx_usr.alias("LoanTransactionUser"),
            Tun.DEP_TRX_USR.alias("DepositTransactionUser"),
            Tun.lns_trx_sn.alias("LoanTransactionSN"),
            Tun.DEP_TRX_SN.alias("DepositTransactionSN"),
            Tun.LNS_TRX_DATE.alias("LoanTransactionDate"),
            Tun.DEP_TRX_DATE.alias("DepositTransactionDate"),
            Tun.DEP_TRANS_SER_NUM.alias("DepositTransactionSerialNumber"),
            Tun.DEP_ENTRY_SER_NUM.alias("DepositEntrySerialNumber"),
            Tun.LNS_TRX_TMSTAMP.alias("TransactionDateTime"),
            Tun.TMSTAMP.alias("TimeStampDate"),
            Tun.businessdate.alias("BusinessDate"),
        )

        result_df_profits.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_tun_profits(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)